In [56]:
#TODO test con s/n minore di zero senza riaddestrare
# TODO Correggi slide 20x10, 40x5
# TODO Aggiungi mlflow

In [57]:
import netCDF4
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta
from tqdm import tqdm
from shapely.geometry import Point
from scipy.stats import skew, kurtosis, entropy
from scipy.fft import fft
from sklearn.preprocessing import MinMaxScaler
import os
from pycaret.classification import setup, compare_models, tune_model, finalize_model, save_model, plot_model, evaluate_model, dashboard, save_experiment, blend_models, get_config
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

In [58]:
ROOT_DIR = 'E:/data/RONGOWAI_L1_SDR_V1.0/' # Change this to your root directory

In [59]:

class SurfaceTypeUtils:
    surface_type_dict = {
        -1: "Ocean",
        0: "NaN",
        1: "Artifical",
        2: "Barely vegetated",
        3: "Inland water",
        4: "Crop",
        5: "Grass",
        6: "Shrub",
        7: "Forest"
    }
    ddm_antennas = {
        0: 'None',
        1: 'Zenith',
        2: 'LHCP',
        3: 'RHCP',
    }

In [60]:

class GeoUtils:
    def __init__(self, world_shapefile_path):
        self.world = gpd.read_file(world_shapefile_path)

    @staticmethod
    def add_seconds(time, seconds):
        timestamp = datetime.strptime(time, "%Y-%m-%d %H:%M:%S")
        new_timestamp = timestamp + timedelta(seconds=seconds)
        return new_timestamp.strftime("%Y-%m-%d %H:%M:%S")

    def is_land(self, lat, lon):
        point = Point(lon, lat)
        return any(self.world.contains(point))

    @staticmethod
    def check_ocean_and_land(lst):
        has_ocean = -1 in lst
        has_land = any(1 <= num <= 7 for num in lst)
        return has_ocean and has_land

    @staticmethod
    def fill_and_filter(arr):
        mask_all_nan = np.all(np.isnan(arr), axis=(2, 3))
        arr_filled = arr.copy()
        for i in range(arr.shape[0]):
            nan_indices = np.where(mask_all_nan[i])[0]
            if len(nan_indices) > 0:
                valid_indices = np.where(~mask_all_nan[i])[0]
                if len(valid_indices) > 0:
                    mean_matrix = np.nanmean(arr[i, valid_indices, :, :], axis=0)
                    arr_filled[i, nan_indices, :, :] = mean_matrix
        mask_discard = np.all(mask_all_nan, axis=1)
        arr_filtered = arr_filled[~mask_discard]
        return arr_filtered, list(np.where(mask_discard.astype(int) == 1)[0])


#### Funzione per il preprocessamento dei dati

In [61]:
class NetCDFPreprocessor:
    def __init__(self, root_dir, preprocessing_method=str):
        self.root_dir = root_dir
        self.netcdf_file_list = os.listdir(root_dir)
        self.preprocessing_method = preprocessing_method
        if self.preprocessing_method not in ['filtered', 'with_lat_lons', 'unfiltered']:
            raise ValueError("Invalid preprocessing method. Choose from 'filtered', 'with_lat_lons', or 'unfiltered'.")

    @staticmethod
    def check_integrity(f):
        """Check integrity of the netCDF file"""
        if not isinstance(f, netCDF4.Dataset):
            raise ValueError("Input must be a netCDF4.Dataset object")
        if 'raw_counts' not in f.variables:
            raise KeyError("The netCDF file does not contain 'raw_counts' variable")
        if 'sp_alt' not in f.variables or 'sp_inc_angle' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_alt' or 'sp_inc_angle' variables")
        if 'sp_rx_gain_copol' not in f.variables or 'sp_rx_gain_xpol' not in f.variables or 'ddm_snr' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_rx_gain_copol', 'sp_rx_gain_xpol' or 'ddm_snr' variables")
        if 'sp_lat' not in f.variables or 'sp_lon' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_lat' or 'sp_lon' variables")
        if 'sp_surface_type' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_surface_type' variable")
        if 'ac_alt' not in f.variables:
            raise KeyError("The netCDF file does not contain 'ac_alt' variable")
        if f.variables['raw_counts'].ndim != 4:
            raise ValueError("The 'raw_counts' variable must have 4 dimensions")

    def preprocess(self, f):
        """ Preprocess the netCDF file and return fit data and labels """
        # Check integrity of the netCDF file
        self.check_integrity(f)

        raw_counts = f.variables['raw_counts'][:]
        ac_alt = f.variables['ac_alt'][:]
        sp_alt = f.variables['sp_alt'][:]
        copol = f.variables['sp_rx_gain_copol'][:]
        xpol = f.variables['sp_rx_gain_xpol'][:]
        snr = f.variables['ddm_snr'][:]
        sp_inc_angle = f.variables['sp_inc_angle'][:]

        distance_2d = (ac_alt[:, np.newaxis] - sp_alt) / np.cos(np.deg2rad(sp_inc_angle)) # Distance between the aircraft and the specular point

        # Filtering mask
        keep_mask = (
            (copol >= 5) & # # SP copolarized gain
            (xpol >= 5) & # SP cross-polarized gain
            (snr > 0) & # Positive signal-to-Noise Ratio
            (distance_2d >= 2000) & #SP distance min
            (distance_2d <= 10000) & #SP distance max
            ~np.isnan(copol) & 
            ~np.isnan(xpol) & 
            ~np.isnan(snr) & 
            ~np.isnan(distance_2d)
        )

        output_array = np.full(raw_counts.shape, np.nan, dtype=np.float32)
        i_indices, j_indices = np.where(keep_mask)
        output_array[i_indices, j_indices] = raw_counts[i_indices, j_indices]

        n_time, n_samples = raw_counts.shape[:2]
        raw_counts_reshaped = output_array.reshape(n_time * n_samples, *raw_counts.shape[2:])
        del output_array

        # Filter out NaN and zero-sum rows
        valid_mask = ~np.any(np.isnan(raw_counts_reshaped), axis=(1, 2)) & (np.sum(raw_counts_reshaped, axis=(1, 2)) > 0)
        fit_data = raw_counts_reshaped[valid_mask].reshape(valid_mask.sum(), -1)

        surface_types = np.nan_to_num(f.variables["sp_surface_type"][:], nan=0).ravel()
        label_data = np.isin(surface_types, np.arange(1, 8)).astype(np.int32)
        label_data = label_data[valid_mask]

        # Ensure that fit_data and label_data have the same length
        assert fit_data.shape[0] == len(label_data), \
            f"Shape mismatch: fit_data {fit_data.shape[0]}, label_data {len(label_data)}"

        return fit_data, label_data

    def preprocess_w_lat_lons(self, f):
        """ Version of the preprocessing function returning latitude and longitudes of the specular points """

        self.check_integrity(f)
        raw_counts = np.array(f.variables['raw_counts'])

        # Distance between the aircraft and the specular point
        ac_alt_2d = np.repeat(np.array(f.variables['ac_alt'])[:, np.newaxis], 20, axis=1)
        distance_2d = (ac_alt_2d - f.variables['sp_alt'][:]) / np.cos(np.deg2rad(f.variables['sp_inc_angle'][:]))

        copol = f.variables['sp_rx_gain_copol'][:]
        xpol = f.variables['sp_rx_gain_xpol'][:]
        snr = f.variables['ddm_snr'][:]
        dist = distance_2d[:]
        specular_point_lat = f.variables['sp_lat'][:]
        specular_point_lon = f.variables['sp_lon'][:]

        # Filtering with mask
        keep_mask = (copol >= 5) & (xpol >= 5) & (snr > 0) & ((dist >= 2000) & (dist <= 10000)) & (~np.isnan(copol.data) & ~np.isnan(xpol.data) & ~np.isnan(snr.data) & ~np.isnan(dist.data) & ~np.isnan(specular_point_lat.data) & ~np.isnan(specular_point_lon.data))
        to_keep_indices = np.argwhere(keep_mask)

        filtered_raw_counts = [raw_counts[i, j] for i, j in to_keep_indices]
        output_array = np.full(raw_counts.shape, np.nan, dtype=np.float32)

        specular_point_lats = specular_point_lat[to_keep_indices[:, 0]]
        specular_point_lons = specular_point_lon[to_keep_indices[:, 0]]

        for idx, (i, j) in enumerate(to_keep_indices):
            output_array[i, j] = filtered_raw_counts[idx]
        # Reshape the output array to match the original dimensions
            raw_counts_filtered = output_array.copy()

        raw_counts_filtered = output_array.copy()
        del output_array

        ddm_data_dict = {
            'Raw_Counts': raw_counts_filtered.reshape(raw_counts_filtered.shape[0]*raw_counts_filtered.shape[1], raw_counts_filtered.shape[2], raw_counts_filtered.shape[3]),
        }
        keep_indices = np.where(
            np.all(~np.isnan(ddm_data_dict['Raw_Counts']), axis=(1, 2)) & (np.sum(ddm_data_dict['Raw_Counts'], axis=(1, 2)) > 0)
        )[0]
        fit_data = np.array([ddm_data_dict['Raw_Counts'][f].ravel() for f in keep_indices])

        specular_point_lats = specular_point_lat.ravel()[keep_indices]
        specular_point_lons = specular_point_lon.ravel()[keep_indices]

        surface_types = f.variables["sp_surface_type"][:]
        surface_types = np.nan_to_num(surface_types, nan=0)
        surface_types_unravelled = surface_types.ravel()
        label_data = [1 if surface_type in np.arange(1, 8) else 0 for surface_type in surface_types_unravelled]
        label_data = [label_data[lab] for lab in range(len(label_data)) if lab in keep_indices]

        assert np.array(fit_data).shape[0] == len(label_data) == np.array(specular_point_lats).shape[0] == np.array(specular_point_lons).shape[0], \
            f"Shape mismatch: fit_data {np.array(fit_data).shape[0]}, label_data {len(label_data)}, lats {np.array(specular_point_lats).shape[0]}, lons {np.array(specular_point_lons).shape[0]}"


        return fit_data, label_data, specular_point_lats, specular_point_lons

    def preprocess_snr_unfiltered(self, f):
        """ Preprocess the netCDF file and return fit data and labels without filtering on signal-to-noise ratio """
        # Check integrity of the netCDF file
        self.check_integrity(f)

        raw_counts = f.variables['raw_counts'][:]
        ac_alt = f.variables['ac_alt'][:]
        sp_alt = f.variables['sp_alt'][:]
        sp_inc_angle = f.variables['sp_inc_angle'][:]
        copol = f.variables['sp_rx_gain_copol'][:]
        xpol = f.variables['sp_rx_gain_xpol'][:]
        #snr = f.variables['ddm_snr'][:]
        
        #Distance between the aircraft and the specular point
        distance_2d = (ac_alt[:, np.newaxis] - sp_alt) / np.cos(np.deg2rad(sp_inc_angle))
        # Filtering mask without SNR
        keep_mask = (
            (copol >= 5) & 
            (xpol >= 5) & 
        #   (snr > 0)  &
            (distance_2d >= 2000) & 
            (distance_2d <= 10000) &
            ~np.isnan(copol) & 
            ~np.isnan(xpol) & 
            #~np.isnan(snr) & 
            ~np.isnan(distance_2d)
        )

        output_array = np.full(raw_counts.shape, np.nan, dtype=np.float32)
        i_indices, j_indices = np.where(keep_mask)
        output_array[i_indices, j_indices] = raw_counts[i_indices, j_indices]

        n_time, n_samples = raw_counts.shape[:2]
        raw_counts_reshaped = output_array.reshape(n_time * n_samples, *raw_counts.shape[2:])
        del output_array
        valid_mask = ~np.any(np.isnan(raw_counts_reshaped), axis=(1, 2)) & (np.sum(raw_counts_reshaped, axis=(1, 2)) > 0)
        fit_data = raw_counts_reshaped[valid_mask].reshape(valid_mask.sum(), -1)

        surface_types = np.nan_to_num(f.variables["sp_surface_type"][:], nan=0).ravel()
        label_data = np.isin(surface_types, np.arange(1, 8)).astype(np.int32)
        label_data = label_data[valid_mask]
        # Ensure that fit_data and label_data have the same length
        assert fit_data.shape[0] == len(label_data), \
            f"Shape mismatch: fit_data {fit_data.shape[0]}, label_data {len(label_data)}"

        return fit_data, label_data


    def process_all_files_random_picked(self, chunk_size = int, sample_fraction = float, n_files_to_pick= int, remove_chunks= bool):
        """ Process all netCDF files in the directory, randomly picking a specified number of files,
        and save the processed data and labels in chunks."""

        full_data = []
        full_labels = []
        counter = 0

        # Take a random number of netCDF files
        if int(len(self.netcdf_file_list)) > n_files_to_pick: # type: ignore
            np.random.seed(42)
            random_netcdf_selected_files = np.random.choice(self.netcdf_file_list, n_files_to_pick, replace=False) # type: ignore
            print('Selezionati 500 file netCDF casuali dalla lista')
        else:
            random_netcdf_selected_files = self.netcdf_file_list

        for file_name in tqdm(random_netcdf_selected_files, desc="Processing files"):
            if not file_name.endswith('.nc'):
                continue
            try:
                f = netCDF4.Dataset(f'{self.root_dir}{file_name}')
                if self.preprocessing_method == 'unfiltered':
                    data, labels = self.preprocess_snr_unfiltered(f)
                elif self.preprocessing_method == 'with_lat_lons':
                    data, labels, latitudes, longitudes = self.preprocess_w_lat_lons(f)
                else:
                    # Default to filtered preprocessing
                    data, labels = self.preprocess(f)
                assert (len(data) == len(labels)), f"Data and labels length mismatch in file {file_name}: {len(data)} != {len(labels)}"
                full_data.append(data)
                full_labels.append(labels)
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")
                continue
            counter += 1
            if counter == n_files_to_pick:
                break
        print(f"Processed {counter} files out of {len(random_netcdf_selected_files)} selected files.")
        # Filtering on data shape
        valid_indices = [i for i, arr in enumerate(full_data) if arr.ndim == 2 if arr.shape[1] == 200]
        full_data_clean = [full_data[i] for i in valid_indices]
        full_labels_clean = [full_labels[i] for i in valid_indices]
        print(f"Number of valid data arrays after filtering: {len(full_data_clean)}")
        # Chunking 
        os.makedirs('test_data/binary_classification', exist_ok=True)

        full_data_sampled = []
        full_labels_sampled = []

        num_chunks = int(np.ceil(len(full_data_clean) / chunk_size))  # type: ignore
        print(f"Total number of chunks: {num_chunks}")
        for idx in range(num_chunks):
            start = idx * chunk_size # type: ignore
            end = min((idx + 1) * chunk_size, len(full_data_clean)) # type: ignore
            chunk_data = np.vstack(full_data_clean[start:end])
            chunk_labels = np.hstack(full_labels_clean[start:end])
            if chunk_data.size == 0 or chunk_labels.size == 0:
                print(f"Skipping empty chunk {idx + 1}/{num_chunks}")
                continue
            print(f"Chunk {idx + 1}/{num_chunks} processed with shape {chunk_data.shape} and labels shape {chunk_labels.shape}")

            # Save each chunk to parquet files
            if chunk_data.shape[0] == 0 or chunk_labels.shape[0] == 0:
                print(f"Skipping empty chunk {idx + 1}/{num_chunks}")
                continue
            fit_data_df = pd.DataFrame(chunk_data)
            labels_df = pd.DataFrame(chunk_labels, columns=['label'])

            table_fit = pa.Table.from_pandas(fit_data_df, preserve_index=False)
            table_labels = pa.Table.from_pandas(labels_df, preserve_index=False)

            pq.write_table(
                table_fit,
                f'test_data/binary_classification/fit_data_chunk_{idx}.parquet',
                compression='zstd',
                use_dictionary=True,
            )
            pq.write_table(
                table_labels,
                f'test_data/binary_classification/labels_chunk_{idx}.parquet',
                compression='zstd',
                use_dictionary=True,
            )
            # Stratified sampling from each chunk
            _, X_sampled, _, y_sampled = train_test_split(
                chunk_data, chunk_labels,
                test_size=sample_fraction,  # type: ignore
                stratify=chunk_labels,
                random_state=42
            )

            full_data_sampled.append(X_sampled)
            full_labels_sampled.append(y_sampled)

        del full_data, full_labels

        full_data_sampled_stratified = np.vstack(full_data_sampled)
        full_labels_sampled_stratified = np.hstack(full_labels_sampled)

        del full_data_sampled, full_labels_sampled
        print(f"Shape of sampled data after chunking and sampling: {np.array(full_data_sampled_stratified).shape}")
        print(f"Shape of sampled labels after chunking and sampling: {np.array(full_labels_sampled_stratified).shape}")

        # Save the final sampled data and labels in parquet format
        if not os.path.exists('test_data/binary_classification'):
            print("Creating directory test_data/binary_classification")
            os.makedirs('test_data/binary_classification', exist_ok=True)

        # Save fit_data 
        fit_data_df = pd.DataFrame(full_data_sampled_stratified)
        table_fit = pa.Table.from_pandas(fit_data_df, preserve_index=False)
        pq.write_table(
            table_fit,
            'test_data/binary_classification/fit_data_binary_test.parquet',
            compression='zstd',
            use_dictionary=True,
        )
        # Save labels
        labels_df = pd.DataFrame(full_labels_sampled_stratified, columns=['label'])
        table_labels = pa.Table.from_pandas(labels_df, preserve_index=False)
        pq.write_table(
            table_labels,
            'test_data/binary_classification/labels_binary_test.parquet',
            compression='zstd',
            use_dictionary=True,
        )
        # Clean up
        del fit_data_df, labels_df, table_fit, table_labels

        print("Data and labels saved in test_data/binary_classification directory.")
        # Remove all chunk parquet files if flag is set (to save space)
        if remove_chunks:
            try:
                chunk_dir = 'test_data/binary_classification'
                for fname in os.listdir(chunk_dir):
                    if fname.startswith('fit_data_chunk_') or fname.startswith('labels_chunk_'):
                        os.remove(os.path.join(chunk_dir, fname))
                print("All chunk files removed.")
            except Exception as e:
                print(f"Error removing chunk files: {e}")

        return full_data_sampled_stratified, full_labels_sampled_stratified

### Funzione per l'estrazione delle features

In [62]:
class DDMFeatureExtractor:
    def __init__(self):
        pass
    def gini(self, array):
            """Gini coefficient calculation"""
            array = np.sort(array)
            index = np.arange(1, array.shape[0] + 1)
            return (np.sum((2 * index - array.shape[0] - 1) * array)) / (array.shape[0] * np.sum(array))  
      
    def extract_ddm_features(self, fit_data: np.ndarray, quadrants = bool) -> pd.DataFrame:
        """
        Extract features from DDM data.
        """
        features = []

        for row in tqdm(fit_data, desc="Extracting DDM features"):
            f = {}
            x = np.array(row, dtype=np.float64) + 1e-10  # evita log(0)

            # 1. General statistics
            f['mean'] = np.mean(x)
            f['std'] = np.std(x)
            f['min'] = np.min(x)
            f['max'] = np.max(x)
            f['median'] = np.median(x)
            f['range'] = np.max(x) - np.min(x)
            f['skew'] = skew(x)
            f['kurtosis'] = kurtosis(x)
            f['entropy'] = entropy(x)
            f['gini'] = self.gini(array=x)

            # 2. Positional 
            f['peak_index'] = np.argmax(x)
            f['peak_value'] = np.max(x)
            f['center_of_mass'] = np.sum(np.arange(len(x)) * x) / np.sum(x)
            f['inertia'] = np.sum(((np.arange(len(x)) - f['center_of_mass'])**2) * x)

            # 3. Segmentations in thirds
            thirds = np.array_split(x, 3)
            for i, part in enumerate(thirds):
                f[f'sum_third_{i+1}'] = np.sum(part)
                f[f'mean_third_{i+1}'] = np.mean(part)
                f[f'max_third_{i+1}'] = np.max(part)

            # 3.1 Segmentations in windows of 5
            windows = np.array_split(x, 5)
            for i, w in enumerate(windows):
                f[f'mean_w{i+1}'] = np.mean(w)
                f[f'std_w{i+1}'] = np.std(w)
                f[f'max_w{i+1}'] = np.max(w)

            # 4. Derivative statistics and differences
            dx = np.diff(x)
            f['mean_diff'] = np.mean(dx)
            f['std_diff'] = np.std(dx)
            f['max_diff'] = np.max(dx)
            f['min_diff'] = np.min(dx)
            f['n_positive_diff'] = np.sum(dx > 0)
            f['n_negative_diff'] = np.sum(dx < 0)
            f['n_zero_diff'] = np.sum(dx == 0)

            # 5. Autocorrelations (lag 1-3)
            for lag in range(1, 4):
                ac = np.corrcoef(x[:-lag], x[lag:])[0, 1] if len(x) > lag else np.nan
                f[f'autocorr_lag{lag}'] = ac

            # 6. FFT 
            spectrum = np.abs(fft(x)) # type: ignore
            half_spectrum = spectrum[:len(spectrum)//2]  
            f['fft_peak_freq'] = np.argmax(half_spectrum)
            f['fft_max'] = np.max(half_spectrum)
            f['fft_median'] = np.median(half_spectrum)
            f['fft_mean'] = np.mean(half_spectrum)

            
            #Quadrant and center statistics
            ddm = row.reshape(10, 20)  # 10x20
            if quadrants: 
                # Quadrants
                q1 = ddm[:5, :10].ravel()
                q2 = ddm[:5, 10:].ravel()
                q3 = ddm[5:, :10].ravel()
                q4 = ddm[5:, 10:].ravel()
                # Quadrante centrale (4x8 centrale)
                center = ddm[3:7, 6:14].ravel()
                # Statistiche dei quadranti
                f['q1_mean'] = np.mean(q1)
                f['q2_mean'] = np.mean(q2)
                f['q3_mean'] = np.mean(q3)
                f['q4_mean'] = np.mean(q4)
                f['center_mean'] = np.mean(center)
                f['q1_std'] = np.std(q1)
                f['q2_std'] = np.std(q2)
                f['q3_std'] = np.std(q3)
                f['q4_std'] = np.std(q4)
                f['center_std'] = np.std(center)
                f['q1_min'] = np.min(q1)
                f['q2_min'] = np.min(q2)
                f['q3_min'] = np.min(q3)
                f['q4_min'] = np.min(q4)
                f['center_min'] = np.min(center)
                f['q1_max'] = np.max(q1)
                f['q2_max'] = np.max(q2)
                f['q3_max'] = np.max(q3)
                f['q4_max'] = np.max(q4)
                f['center_max'] = np.max(center)
                f['q1_median'] = np.median(q1)
                f['q2_median'] = np.median(q2)
                f['q3_median'] = np.median(q3)
                f['q4_median'] = np.median(q4)
                f['center_median'] = np.median(center)
                f['q1_range'] = np.max(q1) - np.min(q1)
                f['q2_range'] = np.max(q2) - np.min(q2)
                f['q3_range'] = np.max(q3) - np.min(q3)
                f['q4_range'] = np.max(q4) - np.min(q4)
                f['center_range'] = np.max(center) - np.min(center)
                f['q1_skew'] = skew(q1)
                f['q2_skew'] = skew(q2)
                f['q3_skew'] = skew(q3)
                f['q4_skew'] = skew(q4)
                f['center_skew'] = skew(center)
                f['q1_kurtosis'] = kurtosis(q1)
                f['q2_kurtosis'] = kurtosis(q2)
                f['q3_kurtosis'] = kurtosis(q3)
                f['q4_kurtosis'] = kurtosis(q4)
                f['center_kurtosis'] = kurtosis(center)
                f['q1_entropy'] = entropy(q1 + 1e-10)
                f['q2_entropy'] = entropy(q2 + 1e-10)
                f['q3_entropy'] = entropy(q3 + 1e-10)
                f['q4_entropy'] = entropy(q4 + 1e-10)
                f['center_entropy'] = entropy(center + 1e-10)
                f['q1_gini'] = self.gini(array=q1)
                f['q2_gini'] = self.gini(array=q2)
                f['q3_gini'] = self.gini(array=q3)
                f['q4_gini'] = self.gini(array=q4)
                f['center_gini'] = self.gini(array=center)

                # Confrontations between quadrants and center 
                
                # Mean differences between quadrants and center
                f['q1_center_mean_diff'] = f['q1_mean'] - f['center_mean']
                f['q2_center_mean_diff'] = f['q2_mean'] - f['center_mean']
                f['q3_center_mean_diff'] = f['q3_mean'] - f['center_mean']
                f['q4_center_mean_diff'] = f['q4_mean'] - f['center_mean']

                # Standard deviation differences between quadrants and center
                f['q1_center_std_diff'] = f['q1_std'] - f['center_std']
                f['q2_center_std_diff'] = f['q2_std'] - f['center_std']
                f['q3_center_std_diff'] = f['q3_std'] - f['center_std']
                f['q4_center_std_diff'] = f['q4_std'] - f['center_std']

                # Maximum differences between quadrants and center
                f['q1_center_max_diff'] = f['q1_max'] - f['center_max']
                f['q2_center_max_diff'] = f['q2_max'] - f['center_max']
                f['q3_center_max_diff'] = f['q3_max'] - f['center_max']
                f['q4_center_max_diff'] = f['q4_max'] - f['center_max']

                # Minimum differences between quadrants and center
                f['q1_center_min_diff'] = f['q1_min'] - f['center_min']
                f['q2_center_min_diff'] = f['q2_min'] - f['center_min']
                f['q3_center_min_diff'] = f['q3_min'] - f['center_min']
                f['q4_center_min_diff'] = f['q4_min'] - f['center_min']

                # entropy differences between quadrants and center
                f['q1_center_entropy_diff'] = f['q1_entropy'] - f['center_entropy']
                f['q2_center_entropy_diff'] = f['q2_entropy'] - f['center_entropy']
                f['q3_center_entropy_diff'] = f['q3_entropy'] - f['center_entropy']
                f['q4_center_entropy_diff'] = f['q4_entropy'] - f['center_entropy']

                # Gini differences between quadrants and center
                f['q1_center_gini_diff'] = f['q1_gini'] - f['center_gini']
                f['q2_center_gini_diff'] = f['q2_gini'] - f['center_gini']
                f['q3_center_gini_diff'] = f['q3_gini'] - f['center_gini']
                f['q4_center_gini_diff'] = f['q4_gini'] - f['center_gini']

                # Skewness differences between quadrants and center
                f['q1_center_skew_diff'] = f['q1_skew'] - f['center_skew']
                f['q2_center_skew_diff'] = f['q2_skew'] - f['center_skew']
                f['q3_center_skew_diff'] = f['q3_skew'] - f['center_skew']
                f['q4_center_skew_diff'] = f['q4_skew'] - f['center_skew']

                # Kurtosis differences between quadrants and center
                f['q1_center_kurtosis_diff'] = f['q1_kurtosis'] - f['center_kurtosis']
                f['q2_center_kurtosis_diff'] = f['q2_kurtosis'] - f['center_kurtosis']
                f['q3_center_kurtosis_diff'] = f['q3_kurtosis'] - f['center_kurtosis']
                f['q4_center_kurtosis_diff'] = f['q4_kurtosis'] - f['center_kurtosis']

            features.append(f)
        return features # type: ignore

### Model trainer 
#### Classe che permette di testare modelli con Pycaret e tunare il migliore, oppure di creare un ensemble di più modelli

In [63]:

class ModelTrainer:
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.final_model = None

    def visualize_model_performances(self, model):

        try:
            print("Valutazione del modello...")
            evaluate_model(model)
        except Exception as e:
            print(f"Errore durante la valutazione del modello: {e}")
        
        try:
            print("Creazione della matrice di confusione del modello...")
            plot_model(model, plot='confusion_matrix', save=True)
            plot_model(model, plot='confusion_matrix', save=False)
        except Exception as e:
            print(f"Errore durante la creazione della matrice di confusione: {e}")

        try:
            print("Creazione del grafico delle feature del modello...")
            plot_model(model, plot='feature_all', save=True)
            plot_model(model, plot='feature_all', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del grafico delle feature: {e}")
        
        try:
            print("Creazione del grafico delle feature del modello (top 20)...")
            plot_model(model, plot='feature', save=True)
            plot_model(model, plot='feature', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del grafico delle feature (top 20): {e}")
        
        try:
            print("Creazione del grafico pipeline del modello...")
            plot_model(model, plot='pipeline', save=True)
            plot_model(model, plot='pipeline', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del grafico pipeline: {e}")

        try:
            print("Creazione curva auc...")
            plot_model(model, plot='auc', save=True)
            plot_model(model, plot='auc', save=False)
        except Exception as e:
            print(f"Errore durante la creazione della curva AUC: {e}")
        
        try:
            print("Creazione del grafico di vc...")
            plot_model(model, plot='vc', save=True)
            plot_model(model, plot='vc', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del grafico di VC: {e}")

        try:
            print("Creazione del report di classificazione del modello...")
            plot_model(model, plot='class_report', save=True)
            plot_model(model, plot='class_report', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del report di classificazione: {e}")
        
        try:
            print("Creazione del grafico PR del modello...")
            plot_model(model, plot='pr', save=True)
            plot_model(model, plot='pr', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del grafico PR: {e}")
        
        try:
            print("Calibrazione del modello...")
            plot_model(model, plot='calibration', save=True)
            plot_model(model, plot='calibration', save=False)
        except Exception as e:
            print(f"Errore durante la creazione del grafico di calibrazione: {e}")


    def search_and_train_single_model(self, model_search=True, n_sample_per_class=int):
        os.environ["PYCARET_CUSTOM_LOGGING_LEVEL"] = "CRITICAL"
 
        if n_sample_per_class <= 0:
            features_df = self.data.reset_index(drop=True)
            labels_df = self.labels.reset_index(drop=True)
            print("No sampling done, using all data.")
        else:
            sampled_indices = (
                self.labels.groupby(self.labels.iloc[:, 0])
                .apply(lambda x: x.sample(n=n_sample_per_class, random_state=42))
                .index.get_level_values(1)
            )
            features_df = self.data.loc[sampled_indices].reset_index(drop=True)
            labels_df = self.labels.loc[sampled_indices].reset_index(drop=True)
            try:
                print(f"Training data dimensions: {features_df.shape}")
                print(f"Labels dimension: {labels_df.shape}")
            except Exception as e:
                print(f"{e}")

        if model_search:
            scaler = MinMaxScaler()
            fit_data_scaled = scaler.fit_transform(features_df)
            clf_exp = setup(data=fit_data_scaled,
                        target=labels_df['0'],
                        #pca=True,
                        #pca_method='incremental',
                        use_gpu=True,
                        feature_selection=True,
                        n_features_to_select=.4,
                        )
            best_models = compare_models(n_select=3, 
                                         exclude=['gbc', 'dummy', 'qda', 'lda', 'nb', 'svm'], # Exclude slowest models 
                                         sort='Accuracy',
                                         )

            print(f"Best model is: {best_models[0]}")
            
            print("Fine tuning the best model...")
            tuned_model = tune_model(best_models[0],
                                    optimize='Accuracy',
                                    n_iter=10,
                                    search_library='optuna',
                                    search_algorithm='tpe',
                                    choose_better=True)
            print("Trained model evalutation:")

            best_params = tuned_model.get_params()

            print("Best hyperparameters:")
            for param, value in best_params.items():
                print(f"{param}: {value}")

            self.final_model = finalize_model(tuned_model)

            # Saving trained model
            save_model(self.final_model, 'best_binary_classification_model')
            print("Final model saved as 'best_binary_classification_model'.")

            try:
                get_config("pipeline")
            except Exception as e:
                print(f"Error during config data retrieval: {e}")
            else:
                print("Pipeline configuration correctly saved.")

            # Salva l'esperimento
            save_experiment('binary_classification_experiment')
            print("Experiment saved as 'binary_classification_experiment'.")
            self.visualize_model_performances(self.final_model)
      
        
    def train_ensemble_model(self, n_sample_per_class=int):  
        os.environ["PYCARET_CUSTOM_LOGGING_LEVEL"] = "CRITICAL"

        if n_sample_per_class <= 0:
            features_df = self.data.reset_index(drop=True)
            labels_df = self.labels.reset_index(drop=True)
            print("No sampling done, using all data.")
        else:
            sampled_indices = (
                self.labels.groupby(self.labels.iloc[:, 0])
                .apply(lambda x: x.sample(n=n_sample_per_class, random_state=42))
                .index.get_level_values(1)
            )
            features_df = self.data.loc[sampled_indices].reset_index(drop=True)
            labels_df = self.labels.loc[sampled_indices].reset_index(drop=True)
            try:
                print(f"Training data dimensions: {features_df.shape}")
                print(f"Labels dimensions: {labels_df.shape}")
            except Exception as e:
                print(f"{e}")

        scaler = MinMaxScaler()
        fit_data_scaled = scaler.fit_transform(features_df)
        clf_exp = setup(data=fit_data_scaled,
                    target=labels_df['0'],
                    #pca=True,
                    #pca_method='incremental',
                    use_gpu=True,
                    feature_selection=True,
                    n_features_to_select=.4,
                    )
        
        best_models = compare_models(n_select=3, 
                                        exclude=['gbc', 'dummy', 'qda', 'lda', 'nb', 'svm'], # Exclude slowest models
                                        sort='Accuracy',
                                        )  
        
        print("Ensembling the best models...")
        best_models = [model for model in best_models if model is not None]
        print(f"Model selected for ensembling: {best_models}")
        ensembled_models = blend_models(best_models, 
                                        method='soft', 
                                        fold=5, 
                                        optimize='Accuracy', 
                                        )
        if ensembled_models is None:
            print("No ensembled models created. Please check the model selection and blending process.")
            return
        try:
            get_config("pipeline")
        except Exception as e:
            print(f"Error during the retrieval of the configurations info: {e}")
        else:
            print("Pipeline config correcly saved.")

        self.visualize_model_performances(ensembled_models)

        self.final_ensembled_model = finalize_model(ensembled_models)
        save_model(self.final_ensembled_model, 'best_binary_classification_ensembled_model')
        save_experiment('binary_classification_ensembled_experiment')

### Preprocessing e features extraction 

In [64]:
read_from_backup = False
if read_from_backup:
    
    fit_data_pl = pd.read_parquet('./processed_data/binary_classification/fit_data_stratified_binary.parquet')
    labels_pl = pd.read_parquet('./processed_data/binary_classification/labels_stratified_binary.parquet')
    fit_data = fit_data_pl.to_numpy()
    labels = labels_pl['label'].to_numpy()
    del fit_data_pl, labels_pl
else:
    preprocessor = NetCDFPreprocessor(root_dir=ROOT_DIR, preprocessing_method='unfiltered')
    fit_data, labels = preprocessor.process_all_files_random_picked( chunk_size = 250, sample_fraction = 0.25, n_files_to_pick= 1000, remove_chunks= True )

Selezionati 500 file netCDF casuali dalla lista


Processing files:   4%|▍         | 39/1000 [00:05<02:42,  5.90it/s]

Error processing file 20250212-195305_NZAA-NZWR_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)


Processing files:  16%|█▋        | 164/1000 [00:24<01:44,  8.02it/s]

Error processing file 20240927-100501_NZHK-NZCH_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)


...

Error processing file 20241219-091144_NZWN-NZNR_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)


Processing files:  93%|█████████▎| 930/1000 [02:30<00:11,  6.03it/s]

Error processing file 20240913-110326_NZRO-NZWN_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)


Processing files: 100%|██████████| 1000/1000 [02:42<00:00,  6.15it/s]


Processed 990 files out of 1000 selected files.
Number of valid data arrays after filtering: 990
Total number of chunks: 4
Chunk 1/4 processed with shape (6280148, 200) and labels shape (6280148,)
Chunk 2/4 processed with shape (6267190, 200) and labels shape (6267190,)
Chunk 3/4 processed with shape (6221175, 200) and labels shape (6221175,)
Chunk 4/4 processed with shape (6057781, 200) and labels shape (6057781,)
Shape of sampled data after chunking and sampling: (6206575, 200)
Shape of sampled labels after chunking and sampling: (6206575,)
Data and labels saved in test_data/binary_classification directory.
All chunk files removed.


In [65]:
# Select a random subset of 2 million samples
num_samples = 2000000
indices = np.random.choice(fit_data.shape[0], size=num_samples, replace=False)
fit_data = fit_data[indices]
labels = labels[indices]

In [66]:
# Instantiate the feature extractor

features_extractor = DDMFeatureExtractor()

In [67]:
from joblib import Parallel, delayed

def extract_ddm_features_row(row):
    return features_extractor.extract_ddm_features(fit_data=np.array([row]), quadrants=False)

combined_features = Parallel(n_jobs=12, backend="loky")(delayed(extract_ddm_features_row)(row) for row in tqdm(fit_data, desc="Estrazione features"))

Estrazione features: 100%|██████████| 2000000/2000000 [04:22<00:00, 7625.33it/s]


In [68]:
flat_features = [row[0] if isinstance(row, list) and len(row) > 0 else row for row in combined_features]
FEATURES = list(flat_features[0].keys())
del combined_features

combined_features = np.array([[row[key] for key in FEATURES] for row in flat_features])
del flat_features
combined_features.shape

# Check for NaN and infinite values
mask_finite = np.isfinite(combined_features).all(axis=1) & (np.abs(combined_features) < np.finfo(np.float64).max).all(axis=1)

fit_data_with_features_clean = combined_features[mask_finite]
labels_clean = labels[mask_finite]
del labels, fit_data

In [69]:
# Saving features and labels
save = True
if save:
    os.makedirs('processed_data/binary_classification/data_w_features', exist_ok=True)
    pd.DataFrame(fit_data_with_features_clean, columns=FEATURES).to_parquet('processed_data/binary_classification/data_w_features/combined_features_no_q.parquet', index=False)
    pd.DataFrame(labels_clean).to_parquet('processed_data/binary_classification/data_w_features/labels_binary_no_q.parquet', index=False)

In [ ]:
# Read the saved features and labels
features_df = pd.read_parquet('processed_data/binary_classification/data_w_features/combined_features.parquet')
labels_df = pd.read_parquet('processed_data/binary_classification/data_w_features/labels_binary.parquet')

In [ ]:
fit_data_with_features_df = pd.DataFrame(fit_data_with_features_clean, columns=FEATURES)
labels_clean_df = pd.DataFrame(labels_clean, columns=['0'])
del fit_data_with_features_clean
fit_data_with_features_df.head()

### Ricerca del miglior modello, modello singolo

In [ ]:
model_trainer = ModelTrainer(data=fit_data_with_features_df, labels=labels_clean_df)
model_trainer.search_and_train_single_model(model_search=True, n_sample_per_class=250000)

In [ ]:
#load_model = load_model('best_binary_classification_model')


### Training dell'ensemble dei modelli migliori

In [ ]:
model_trainer = ModelTrainer(data=features_df, labels=labels_df)
model_trainer.train_ensemble_model(n_sample_per_class=200000)